# Iteration 04 - Calibration And Comparison

## Aim

Calibrate the occupancy-audit interpretation against the published tables in Monks et al., compare current-admissions delay estimates for acute and rehab beds, and produce a delay trade-off chart comparable in purpose to Figure 3.


## Prompt Used

In iteration 04, calibrate and compare the audited occupancy and p(delay) outputs against the published tables.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stroke_sim.config import SimulationSettings
from stroke_sim.metrics import plot_delay_tradeoff_curve
from stroke_sim.runner import build_delay_tradeoff, run_replications
from stroke_sim.validation import (
    PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
    PUBLISHED_CURRENT_ADMISSIONS_REHAB,
    compare_to_published_table,
)


## Replication Run

This uses the paper's run length and warm-up, with a reduced replication count for iterative development speed.


In [ ]:
settings = SimulationSettings(run_length_days=365 * 5, warm_up_days=365 * 3, replications=30)
audit = run_replications(settings=settings)
audit.head()


## Published Comparison Tables

The paper reports current-admissions delay values for multiple acute and rehab bed counts. Here we compare the audited Erlang-style delay estimates against those published values.


In [ ]:
acute_comparison = compare_to_published_table(
    audit,
    column='acute_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
)
rehab_comparison = compare_to_published_table(
    audit,
    column='rehab_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_REHAB,
)

acute_comparison


In [ ]:
rehab_comparison


## Acute Delay Trade-Off Curve

This chart mirrors the role of the paper's Figure 3, showing the trade-off between acute bed numbers and probability of delay.


In [ ]:
acute_tradeoff = build_delay_tradeoff(audit, column='acute_occupancy', bed_range=range(0, 29))
tradeoff_chart_path = root / 'docs' / 'figures' / 'iteration_04_acute_delay_tradeoff.png'
fig = plot_delay_tradeoff_curve(
    acute_tradeoff,
    title='Simulated trade-off between the probability that a patient is delayed and the no. of acute beds available',
    x_label='No. of acute beds available',
    output_path=tradeoff_chart_path,
)
fig


## Summary Metrics


In [ ]:
print({
    'acute_mean_abs_error': float(acute_comparison['abs_error'].mean()),
    'rehab_mean_abs_error': float(rehab_comparison['abs_error'].mean()),
})


## Saved Figure

- `docs/figures/iteration_04_acute_delay_tradeoff.png`


## Tester Notes

- The main calibration change in this iteration is interpretive rather than structural: delay is estimated using the Erlang-loss-style occupancy calculation described in the paper.
- This brings the current-admissions acute and rehab comparisons materially closer to the published table values than the naive `P(occupancy >= beds)` estimate.
- The model is still a simplified reconstruction, so remaining mismatch should be expected and documented.
